# Spår A – Regression (förutsäga bostadsvärde)

## Bakgrundshistoria
En kund vill kunna göra grova värderingar av bostadsområden för att kunna planera investeringar och resurser. De vill kunna uppskatta bostadsvärdet baserat på områdets egenskaper.

## Uppdrag
Bygg en modell som förutsäger median_house_value.

## Syfte
Att kunna göra en rimlig värdeprognos och samtidigt förstå vilka faktorer som verkar hänga ihop med högre/lägre värde.

# Import

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.dummy import DummyRegressor

from sklearn.metrics import confusion_matrix, classification_report, f1_score, mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error


# Functions

In [ ]:
def ev_model(y_true, y_pred, model_name: str) -> dict:
    """
    Utvärdera en baseline modell med flera vanliga mått.
    Returnerar en dict med RMSE, MAE, R2
    """
    rmse = root_mean_squared_error(y_true=y_true, y_pred=y_pred)
    mae = mean_absolute_error(y_true=y_true, y_pred=y_pred)
    r2 = r2_score(y_true=y_true, y_pred=y_pred)

    return {"model": model_name, "RMSE": rmse, "MAE": mae, "R2": r2}

# 1) Dataförståelse & EDA 

## Data

In [ ]:
file_path = "/housing.csv"
folder_path = "data"

df = pd.read_csv(folder_path + file_path)

print(df.info())
display(df.head())
df.describe()

## Figure

In [ ]:
#Heat map

correlation_matrix = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10,6))
sns.heatmap(correlation_matrix, annot=True, ax=ax)

## Heatmap Tolkning.

- Korrelationen mellan median_income och median_house_value ligger på ungefär 0.69, vilket innebär ett relativt starkt positivt samband.
- Den feature som har näst högst korrelation med median_house_value är total_rooms (≈0.13), vilket är ett svagt samband.

In [ ]:
#Histogram

fig, ax = plt.subplots(figsize=(10,6))
ax.hist(df["median_income"], bins=10)
ax.set_title("Median Income")
ax.set_xlabel("Median Income Value")
ax.set_ylabel("Amount of Incomes")
ax.grid(True, axis="y")
plt.tight_layout()


# Histogram på Median Income Tolkning

- Majoriteten av Incomes har en median_income mellan 2 och 5. Det finns även en del områden mellan 1 och 2 samt mellan 5 och 7, medan mycket få områden har median_income över 8.

# 2) Split + Preprocessing

In [ ]:
target_name = "median_house_value"

X = df.drop(columns=[target_name])
y = df[target_name]

# make numeric/category feature
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["string"]).columns.tolist()


X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.30,
    random_state=42
)



In [ ]:
print("X_Train : ", X_train.shape, "X_Test : ", X_test.shape)
print(f"\n{target_name} is important to have no missing values")
print("\nMissing Values (train)")
print(X_train.isna().sum().sort_values(ascending=False).head(10))
print("\nMissing Values (test)")
print(X_test.isna().sum().sort_values(ascending=False).head(10))
print("numeric_features :", numeric_features)
print("\ncategorical_features :", categorical_features)
print(X["ocean_proximity"].unique())

## Pipeline

In [ ]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore", 
        sparse_output=True
    ))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

logistic_regression_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=100))
])

linear_regression_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

random_forest_classifier_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=42

    ))
])

random_forest_regressor_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=42

    ))
])

In [ ]:
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "neg_root_mean_squared_error"

# 3) Modellering

## Baseline (DummyRegressor)

In [ ]:
# Med en konversation med AI så sa den att datasetet var skevt så mitt val av strategy"median" var 
# bättre än att ta mean. så med dens förklaring att median är bättre för skev data och tack vare
# att hist av median_income var skev så tar jag median som strategy.

baseline_model_median = DummyRegressor(strategy="median")

scores_baseline_median = cross_val_score(
    estimator=baseline_model_median,
    X=X_train,
    y=y_train,
    scoring="neg_root_mean_squared_error",
    cv=cv_strategy
)

print("Baseline (median) RMAE : ", -scores_baseline_median.mean())

baseline_model_mean = DummyRegressor(strategy="mean")

scores_baseline_mean = cross_val_score(
    estimator=baseline_model_mean,
    X=X_train,
    y=y_train,
    scoring="neg_root_mean_squared_error",
    cv=cv_strategy
)

print("Baseline (mean) RMAE : ", -scores_baseline_mean.mean())


## LogisticRegression

In [ ]:
# scores_log = cross_val_score(
#     estimator=logistic_regression_pipeline,
#     X=X_train,
#     y=y_train,
#     scoring="neg_root_mean_squared_error",
#     cv=cv_strategy
# )

# print("LogisticRegression RMAE : ", -scores_log.mean())

# jag gjorde fel först och tänkte mig inte för och tog en klassifer model till ett regression problem.
# låter detta vara kvar som reminder och att see hur man kan veta att man valt fel.

## RandomForestClassifier

In [ ]:
# scores_rfc = cross_val_score(
#     estimator=random_forest_classifier_pipeline,
#     X=X_train,
#     y=y_train,
#     scoring="neg_root_mean_squared_error",
#     cv=cv_strategy
# )

# print("RandomForestClassifer RMAE : ", -scores_rfc.mean())

# jag gjorde fel först och tänkte mig inte för och tog en klassifer model till ett regression problem.
# låter detta vara kvar som reminder och att see hur man kan veta att man valt fel.

## LinerRegression

In [ ]:
scores_lin = cross_val_score(
    estimator=linear_regression_pipeline,
    X=X_train,
    y=y_train,
    scoring="neg_root_mean_squared_error",
    cv=cv_strategy
)

print("LinearRegression RMAE : ", -scores_lin.mean())

## RandomForestRegressor

In [ ]:
scores_rfr = cross_val_score(
    estimator=random_forest_regressor_pipeline,
    X=X_train,
    y=y_train,
    scoring="neg_root_mean_squared_error",
    cv=cv_strategy
)
print("RandomForestRegressor RMAE : ", -scores_rfr.mean())

In [ ]:
best_model_name = "LinearRegression" if -scores_lin.mean() < -scores_rfr.mean() else "RandomForestRegressor"
print("Best Model : ", best_model_name)

# 4) Välj och optimera en modell

In [ ]:
random_forest_regressor_pipeline.get_params().keys()

In [ ]:
rfr = RandomForestRegressor()
rfr.get_params()

In [ ]:
param_grid_rfr = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__min_samples_split": [2, 5, 10]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=random_forest_regressor_pipeline,
    param_grid=param_grid_rfr,
    scoring=SCORING,
    cv=cv_strategy,
    n_jobs=-1
)

In [ ]:
grid_search.fit(X_train, y_train)

print("Best GB params:", grid_search.best_params_)
print("Best CV-RMSE:", -grid_search.best_score_)

### Hyperparameter-tuning (GridSearchCV)

Model vald: RandomForestRegression.

Jag använde GridSearchCV med KFold med n_folds = 5 cross-validation och RMSE som scoring-metric.

Jag optimerade följande hyperparametrar:
- n_estimators (antal träd)
- max_depth (maximalt träd-djup)
- min_samples_split (minsta antal samples för att dela en nod)
- min_samples_leaf (minsta antal samples i ett blad)

Dessa parametrar valdes eftersom de har störst påverkan på modellens generaliseringsförmåga och risk för överfitting.

Tunning modellen fick sämre RMSE för att default‑inställningarna för RandomForest är:
- max_depth = None 
- min_samples_leaf = 1
- min_samples_split = 2
- n_estimators = 100

vilket betyder att jag som gjorde "model__max_depth": [2, 5, 10] tillät den inte att köra none max_depth. om jag byter till "model__max_depth": [2, 5, 10, none] skulle RMSE av tunning antagligen bli bättre.

# 5) Slutlig utvärdering på testdata + rekommendation

In [ ]:
best_model = grid_search.best_estimator_
best_model

In [ ]:
y_pred = best_model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)

mae = mean_absolute_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print("Test RMSE:", rmse)
print("Test MAE:", mae)
print("Test R2:", r2)


### RMSE(Root Mean Squared Error)

Valde RMSE för den straffar stora fel mer än små, vilket eftersom målet med denna rapport är att förutsäga värdet på bostäder vilket ofta är stora summor. så en metric som straffar stora fel är bättre när det gäller stora summor.
RMSE är standard inom regression och prissättningsproblem. 


In [ ]:
pd_results = {
    "BaseLine RMSE": 
}